In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RISK_PATH = Path(
    r"C:\Users\HP\OneDrive\Documents\SIH 26083\data\processed\wards\final_ward_risk_dataset_RF.csv"
)

risk = pd.read_csv(RISK_PATH)

print(risk.shape)
risk.head()

(888750, 13)


,Ward_No,Static_Heat_Score,Vulnerability_Score,Response_Gap_Score,Prediction_Date,Target_Date,Lead_Time_Days,forecast_wbgt_max_C,forecast_wbgt_mean_C,WBGT_Hazard_Score,Future_Heat_Hazard,Human_Heat_Risk,Risk_Category
0,1,0.3992,0.469661,0.972,2024-01-18,2024-01-19,1,18.539018,15.562002,0.017159,0.131771,0.401184,Moderate
1,1,0.3992,0.469661,0.972,2024-01-18,2024-01-20,2,18.585355,15.800021,0.017159,0.131771,0.401184,Moderate
2,1,0.3992,0.469661,0.972,2024-01-18,2024-01-21,3,17.955183,16.066384,0.005626,0.123698,0.397147,Low
3,1,0.3992,0.469661,0.972,2024-01-18,2024-01-22,4,18.976895,15.731051,0.032349,0.142404,0.406500,Moderate
4,1,0.3992,0.469661,0.972,2024-01-18,2024-01-23,5,18.978847,16.285110,0.032349,0.142404,0.406500,Moderate


In [2]:
# ============================================================
# CELL 2 — FINAL DATASET QA
# ============================================================

print("Shape:", risk.shape)

print("\n--- Columns ---")
print(risk.columns.tolist())

print("\n--- Unique wards ---")
print(risk["Ward_No"].nunique())

print("\n--- Lead times ---")
print(sorted(risk["Lead_Time_Days"].unique()))

print("\n--- Date range ---")
print("Prediction:", risk["Prediction_Date"].min(), "to", risk["Prediction_Date"].max())
print("Target:", risk["Target_Date"].min(), "to", risk["Target_Date"].max())

print("\n--- Duplicate ward/date/lead combinations ---")
duplicates = risk.duplicated(
    subset=["Ward_No", "Prediction_Date", "Target_Date", "Lead_Time_Days"]
).sum()

print("Duplicates:", duplicates)

print("\n--- Missing values ---")
print(risk.isna().sum())

print("\n--- Risk range ---")
print("Minimum:", risk["Human_Heat_Risk"].min())
print("Maximum:", risk["Human_Heat_Risk"].max())

Shape: (888750, 13)

--- Columns ---
['Ward_No', 'Static_Heat_Score', 'Vulnerability_Score', 'Response_Gap_Score', 'Prediction_Date', 'Target_Date', 'Lead_Time_Days', 'forecast_wbgt_max_C', 'forecast_wbgt_mean_C', 'WBGT_Hazard_Score', 'Future_Heat_Hazard', 'Human_Heat_Risk', 'Risk_Category']

--- Unique wards ---
250

--- Lead times ---
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

--- Date range ---
Prediction: 2024-01-18 to 2025-12-30
Target: 2024-01-19 to 2025-12-31

--- Duplicate ward/date/lead combinations ---
Duplicates: 0

--- Missing values ---
Ward_No                 0
Static_Heat_Score       0
Vulnerability_Score     0
Response_Gap_Score      0
Prediction_Date         0
Target_Date             0
Lead_Time_Days          0
forecast_wbgt_max_C     0
forecast_wbgt_mean_C    0
WBGT_Hazard_Score       0
Future_Heat_Hazard      0
Human_Heat_Risk         0
Risk_Category           0
dtype: int64

--- Risk range ---
Minimum: 0.1355524447620343
Maximum: 0.8111891960

In [3]:
# ============================================================
# CELL 3 — OVERALL RISK DISTRIBUTION
# ============================================================

# Counts
risk_counts = risk["Risk_Category"].value_counts()

# Put categories in the correct logical order
category_order = [
    "Very Low",
    "Low",
    "Moderate",
    "High",
    "Very High"
]

risk_counts = risk_counts.reindex(category_order, fill_value=0)

# Percentages
risk_percent = (risk_counts / len(risk) * 100).round(2)

risk_distribution = pd.DataFrame({
    "Count": risk_counts,
    "Percentage": risk_percent
})

print("FINAL RISK DISTRIBUTION")
print("=" * 45)
print(risk_distribution)

print("\nOverall risk statistics")
print("=" * 45)
print(risk["Human_Heat_Risk"].describe())

FINAL RISK DISTRIBUTION
                Count  Percentage
Risk_Category                    
Very Low         3051        0.34
Low            183597       20.66
Moderate       502625       56.55
High           199433       22.44
Very High          44        0.00

Overall risk statistics
count    888750.000000
mean          0.500423
std           0.119362
min           0.135552
25%           0.416142
50%           0.502305
75%           0.589845
max           0.811189
Name: Human_Heat_Risk, dtype: float64


In [4]:
# ============================================================
# CELL 4 — HIGH-RISK WARD PRIORITIZATION
# ============================================================

# Identify High and Very High predictions
high_risk = risk[
    risk["Risk_Category"].isin(["High", "Very High"])
].copy()

# Summarize each ward
ward_summary = (
    high_risk
    .groupby("Ward_No")
    .agg(
        High_VeryHigh_Count=("Risk_Category", "size"),
        Average_Risk=("Human_Heat_Risk", "mean"),
        Maximum_Risk=("Human_Heat_Risk", "max"),
        Average_WBGT=("forecast_wbgt_max_C", "mean"),
        Average_Vulnerability=("Vulnerability_Score", "mean"),
        Average_Response_Gap=("Response_Gap_Score", "mean")
    )
    .reset_index()
)

# Total number of forecast scenarios available per ward
total_scenarios = (
    risk
    .groupby("Ward_No")
    .size()
    .rename("Total_Forecast_Instances")
    .reset_index()
)

# Merge
ward_summary = ward_summary.merge(
    total_scenarios,
    on="Ward_No",
    how="left"
)

# Percentage of forecasts classified High/Very High
ward_summary["High_VeryHigh_Percent"] = (
    ward_summary["High_VeryHigh_Count"]
    / ward_summary["Total_Forecast_Instances"]
    * 100
)

# Sort by frequency of High/Very High classification
ward_summary = ward_summary.sort_values(
    "High_VeryHigh_Count",
    ascending=False
)

# Top 20
top_20_wards = ward_summary.head(20).copy()

print("TOP 20 WARDS BY HIGH / VERY HIGH RISK FREQUENCY")
print("=" * 70)

print(
    top_20_wards[
        [
            "Ward_No",
            "High_VeryHigh_Count",
            "High_VeryHigh_Percent",
            "Average_Risk",
            "Maximum_Risk",
            "Average_WBGT",
            "Average_Vulnerability",
            "Average_Response_Gap"
        ]
    ].round(3).to_string(index=False)
)

TOP 20 WARDS BY HIGH / VERY HIGH RISK FREQUENCY
 Ward_No  High_VeryHigh_Count  High_VeryHigh_Percent  Average_Risk  Maximum_Risk  Average_WBGT  Average_Vulnerability  Average_Response_Gap
      34                 2339                 65.795         0.703         0.811        32.274                  0.520                 0.966
      94                 2255                 63.432         0.701         0.806        32.476                  0.865                 0.670
      37                 2254                 63.404         0.700         0.805        32.478                  0.589                 0.872
      44                 2254                 63.404         0.699         0.804        32.478                  0.728                 0.842
     126                 2211                 62.194         0.697         0.800        32.578                  0.374                 1.000
     180                 2176                 61.210         0.694         0.796        32.659                  

In [5]:
# ============================================================
# CELL 5 — NIGHT-BEFORE ALERT ANALYSIS
# ============================================================

night_before = risk[
    risk["Lead_Time_Days"] == 1
].copy()

night_alerts = night_before[
    night_before["Risk_Category"].isin(["High", "Very High"])
].copy()

print("NIGHT-BEFORE FORECASTS")
print("=" * 60)
print("Total forecast instances:", len(night_before))
print("Unique wards:", night_before["Ward_No"].nunique())

print("\nHIGH / VERY HIGH ALERTS")
print("=" * 60)
print("Alert instances:", len(night_alerts))
print(
    "Percentage:",
    round(len(night_alerts) / len(night_before) * 100, 2),
    "%"
)

# Total forecasts per ward
ward_totals = (
    night_before
    .groupby("Ward_No")
    .size()
    .rename("Total_Night_Before_Forecasts")
)

# High-risk alerts per ward
night_ward_summary = (
    night_alerts
    .groupby("Ward_No")
    .agg(
        Alert_Count=("Risk_Category", "size"),
        Average_Risk=("Human_Heat_Risk", "mean"),
        Maximum_Risk=("Human_Heat_Risk", "max"),
        Average_WBGT=("forecast_wbgt_max_C", "mean"),
        Vulnerability=("Vulnerability_Score", "mean"),
        Response_Gap=("Response_Gap_Score", "mean")
    )
)

night_ward_summary = night_ward_summary.join(ward_totals)

night_ward_summary["Alert_Percent"] = (
    night_ward_summary["Alert_Count"]
    / night_ward_summary["Total_Night_Before_Forecasts"]
    * 100
)

night_ward_summary = (
    night_ward_summary
    .reset_index()
    .sort_values("Alert_Count", ascending=False)
)

print("\nTOP 10 WARDS FOR NIGHT-BEFORE ALERTS")
print("=" * 60)

print(
    night_ward_summary.head(10).round(3).to_string(index=False)
)

NIGHT-BEFORE FORECASTS
Total forecast instances: 178250
Unique wards: 250

HIGH / VERY HIGH ALERTS
Alert instances: 40503
Percentage: 22.72 %

TOP 10 WARDS FOR NIGHT-BEFORE ALERTS
 Ward_No  Alert_Count  Average_Risk  Maximum_Risk  Average_WBGT  Vulnerability  Response_Gap  Total_Night_Before_Forecasts  Alert_Percent
      34          472         0.703         0.801        32.285          0.520         0.966                           713         66.199
      37          452         0.702         0.795        32.524          0.589         0.872                           713         63.394
      44          452         0.700         0.794        32.524          0.728         0.842                           713         63.394
      94          452         0.702         0.795        32.524          0.865         0.670                           713         63.394
     126          447         0.697         0.789        32.581          0.374         1.000                           713        

In [6]:
# ============================================================
# CELL 6 — RISK BY FORECAST LEAD TIME
# ============================================================

lead_summary = (
    risk
    .groupby("Lead_Time_Days")
    .agg(
        Average_Risk=("Human_Heat_Risk", "mean"),
        Maximum_Risk=("Human_Heat_Risk", "max"),
        High_VeryHigh_Count=(
            "Risk_Category",
            lambda x: x.isin(["High", "Very High"]).sum()
        ),
        Total_Instances=("Risk_Category", "size")
    )
    .reset_index()
)

lead_summary["High_VeryHigh_Percent"] = (
    lead_summary["High_VeryHigh_Count"]
    / lead_summary["Total_Instances"]
    * 100
)

print("RISK BY FORECAST LEAD TIME")
print("=" * 70)

print(
    lead_summary.round(3).to_string(index=False)
)

RISK BY FORECAST LEAD TIME
 Lead_Time_Days  Average_Risk  Maximum_Risk  High_VeryHigh_Count  Total_Instances  High_VeryHigh_Percent
              1         0.502         0.801                40503           178250                 22.723
              2         0.500         0.805                39562           178000                 22.226
              3         0.500         0.802                39685           177750                 22.326
              4         0.501         0.804                39935           177500                 22.499
              5         0.499         0.811                39792           177250                 22.450


In [7]:
# ============================================================
# CELL 7 — SENSITIVITY ANALYSIS
# ============================================================

scenarios = {
    "Baseline": {
        "hazard": 0.50,
        "vulnerability": 0.30,
        "response": 0.20
    },
    "Hazard_Heavy": {
        "hazard": 0.60,
        "vulnerability": 0.25,
        "response": 0.15
    },
    "Vulnerability_Heavy": {
        "hazard": 0.40,
        "vulnerability": 0.40,
        "response": 0.20
    },
    "Response_Heavy": {
        "hazard": 0.40,
        "vulnerability": 0.25,
        "response": 0.35
    }
}

sensitivity_results = []

for name, weights in scenarios.items():

    scenario_risk = (
        weights["hazard"] * risk["Future_Heat_Hazard"]
        + weights["vulnerability"] * risk["Vulnerability_Score"]
        + weights["response"] * risk["Response_Gap_Score"]
    )

    temp = risk[["Ward_No"]].copy()
    temp["Scenario_Risk"] = scenario_risk

    ward_avg = (
        temp.groupby("Ward_No")["Scenario_Risk"]
        .mean()
        .sort_values(ascending=False)
    )

    top_10_mean = ward_avg.head(10).mean()

    sensitivity_results.append({
        "Scenario": name,
        "Mean_Risk": scenario_risk.mean(),
        "Max_Risk": scenario_risk.max(),
        "Top_Ward": ward_avg.index[0],
        "Top_Ward_Risk": ward_avg.iloc[0],
        "Top_10_Mean_Risk": top_10_mean
    })

sensitivity_df = pd.DataFrame(sensitivity_results)

print("SENSITIVITY ANALYSIS")
print("=" * 80)
print(sensitivity_df.round(4).to_string(index=False))

SENSITIVITY ANALYSIS
           Scenario  Mean_Risk  Max_Risk  Top_Ward  Top_Ward_Risk  Top_10_Mean_Risk
           Baseline     0.5004    0.8112        34         0.6477            0.6353
       Hazard_Heavy     0.5001    0.8309       180         0.6347            0.6233
Vulnerability_Heavy     0.5007    0.8096        94         0.6788            0.6453
     Response_Heavy     0.5007    0.8377        34         0.7069            0.6859


In [8]:
# ============================================================
# CELL 8 — TOP-20 ROBUSTNESS ACROSS SCENARIOS
# ============================================================

scenario_top20 = {}

for name, weights in scenarios.items():

    scenario_risk = (
        weights["hazard"] * risk["Future_Heat_Hazard"]
        + weights["vulnerability"] * risk["Vulnerability_Score"]
        + weights["response"] * risk["Response_Gap_Score"]
    )

    ward_avg = (
        pd.DataFrame({
            "Ward_No": risk["Ward_No"],
            "Scenario_Risk": scenario_risk
        })
        .groupby("Ward_No")["Scenario_Risk"]
        .mean()
        .sort_values(ascending=False)
    )

    scenario_top20[name] = set(ward_avg.head(20).index)

print("TOP-20 WARD ROBUSTNESS")
print("=" * 70)

for name, wards in scenario_top20.items():
    print(f"{name}: {sorted(wards)}")

common_top20 = set.intersection(*scenario_top20.values())

print("\nCOMMON TOP-20 WARDS ACROSS ALL SCENARIOS")
print("=" * 70)
print("Number of common wards:", len(common_top20))
print("Common wards:", sorted(common_top20))

# Pairwise overlap
print("\nPAIRWISE TOP-20 OVERLAP")
print("=" * 70)

scenario_names = list(scenario_top20.keys())

for i in range(len(scenario_names)):
    for j in range(i + 1, len(scenario_names)):

        a = scenario_names[i]
        b = scenario_names[j]

        overlap = len(
            scenario_top20[a].intersection(scenario_top20[b])
        )

        print(f"{a} vs {b}: {overlap}/20")

TOP-20 WARD ROBUSTNESS
Baseline: [20, 32, 34, 37, 38, 41, 43, 44, 46, 49, 50, 94, 108, 123, 125, 126, 127, 179, 180, 240]
Hazard_Heavy: [20, 32, 34, 37, 41, 43, 44, 46, 50, 94, 125, 126, 127, 133, 177, 179, 180, 181, 238, 240]
Vulnerability_Heavy: [32, 34, 37, 38, 41, 42, 43, 44, 45, 46, 49, 50, 67, 94, 108, 123, 126, 179, 180, 240]
Response_Heavy: [2, 29, 30, 31, 32, 34, 35, 37, 38, 41, 43, 44, 46, 49, 108, 123, 125, 126, 127, 129]

COMMON TOP-20 WARDS ACROSS ALL SCENARIOS
Number of common wards: 8
Common wards: [32, 34, 37, 41, 43, 44, 46, 126]

PAIRWISE TOP-20 OVERLAP
Baseline vs Hazard_Heavy: 16/20
Baseline vs Vulnerability_Heavy: 17/20
Baseline vs Response_Heavy: 14/20
Hazard_Heavy vs Vulnerability_Heavy: 13/20
Hazard_Heavy vs Response_Heavy: 10/20
Vulnerability_Heavy vs Response_Heavy: 12/20
